In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
%cd drive/MyDrive/

In [ ]:
!rm -rf FPL_forecast
!git clone https://github.com/bragehs/FPL_forecast.git

In [ ]:
%cd FPL_forecast/predictor/

In [1]:
file_path = '/content/drive/MyDrive/colab_fpl'
file_path

'/content/drive/MyDrive/colab_fpl'

In [1]:
import os
import torch
from training import train_model, hyperparameter_tuning
from model import FPLSequenceModel

In [2]:
file_path = os.getcwd() + "/processed_data"
file_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [3]:
X_train_numeric = torch.load(file_path + "/X_train.pt", weights_only=True)
X_train_static = torch.load(file_path + "/X_static_train.pt", weights_only=True)
y_train = torch.load(file_path + "/y_train.pt", weights_only=True)
minutes_train = torch.load(file_path + "/minutes_train.pt", weights_only=True)
X_val_numeric = torch.load(file_path + "/X_val.pt", weights_only=True)
X_val_static = torch.load(file_path + "/X_static_val.pt", weights_only=True)
y_val = torch.load(file_path + "/y_val.pt", weights_only=True)
minutes_val = torch.load(file_path + "/minutes_val.pt", weights_only=True)

print(f"Train sequences: {X_train_numeric.shape}, {X_train_static.shape}, Targets: {y_train.shape}")

Train sequences: torch.Size([162512, 5, 16]), torch.Size([162512, 19]), Targets: torch.Size([162512, 1])


In [4]:
# Hyperparameter tuning
best_params = hyperparameter_tuning(X_train_numeric=X_train_numeric, y_train=y_train,X_train_static=X_train_static, minutes_train=minutes_train,
                                    X_val_numeric=X_val_numeric, X_val_static=X_val_static, y_val=y_val,
                                    epochs=1, n_trials=1, num_workers=4,
                                    )

Running random search with 1 trials...

Trial 1/1
Params: {'learning_rate': 0.0001, 'hidden_dim': 256, 'weight_decay': 0.0001, 'lstm_layers': 4, 'dropout': 0.4, 'batch_size': 64, 'alpha': 0.5}


libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x1205a1b20>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1603, in __del__
    def __del__(self):

  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/_utils/signal_handling.py", line 73, in handler
    _error_if_any_worker_fails()
RuntimeError: DataLoader worker (pid 41007) is killed by signal: Abort trap: 6. 


KeyboardInterrupt: 

In [ ]:
print("\nTraining final model with best hyperparameters...")
model = FPLSequenceModel(
        numeric_seq_dim=X_train_numeric.shape[-1],
        static_dim=X_train_static.shape[-1],
        hidden_dim=best_params["hidden_dim"],
        lstm_layers=best_params["lstm_layers"],
        dropout=best_params["dropout"],
        multitask=True
    )


Training final model with best hyperparameters...


In [ ]:
# Full training with best hyperparameters
train_model(
    model=model,
    X_train_numeric=X_train_numeric, X_train_static=X_train_static, y_train=y_train, minutes_train=minutes_train,
    X_val_numeric=X_val_numeric, X_val_static=X_val_static, y_val=y_val,
    learning_rate=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"],
    batch_size=best_params["batch_size"],
    alpha=best_params["alpha"],
    epochs=100,
    verbose=2,
    num_workers=4,
        )

Epoch  1/100:   6%|▋         | 321/5079 [00:14<02:55, 27.07it/s, loss=10.2737, lr=0.01]libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x1197a1bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1568, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/multiprocessing/process.py",

KeyboardInterrupt: 